# Final models: inference on the focal-sampling data

Runs the B2 models from `10a_final_models_training` on the focal-sampled data: HR and RF on the uncorrected B2 file, and HR on the rotbasal and rotdaily B2 files. HR B2 rotdaily is the final classifier; the others are comparison models. TabPFN B2 uncorrected, also a comparison model, needs the `tabpfn` environment and is run in `10b_final_models_inference_tabpfn`.

Outputs (`data/output/inference_results/`):
1. `final_hr_b2_unc_focal_sampled_random_predictions.parquet`: HR label and class probabilities per burst
2. `final_rf_b2_unc_focal_sampled_random_predictions.parquet`: RF label and class probabilities per burst
3. `final_hr_b2_rotbasal_focal_sampled_random_predictions.parquet` and `final_hr_b2_rotdaily_focal_sampled_random_predictions.parquet`: same format as 1

In [ ]:
import datetime
import os
import re

import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pytz
import torch
import tsai.all as ts
from tqdm import tqdm

In [ ]:
HR_DIR, HR_NAME = "../models/final_hr_b2_unc", "final_hr_b2_unc"
RF_DIR, RF_NAME = "../models/final_rf_b2_unc", "final_rf_b2_unc"
FOCAL_ACC = "../data/raw/focal_sampling/acc/focal_sampled_acc_burst_2_uncorrected.parquet"
FOCAL_FEATURES = (
    "../data/raw/focal_sampling/features/focal_sampled_features_burst_2_uncorrected.parquet"
)
OUT_DIR = "../data/output/inference_results"
HR_OUT = f"{OUT_DIR}/{HR_NAME}_focal_sampled_random_predictions.parquet"
RF_OUT = f"{OUT_DIR}/{RF_NAME}_focal_sampled_random_predictions.parquet"

CLASS_COLS = [
    "Eating",
    "Grooming actor",
    "Grooming receiver",
    "Resting",
    "Running",
    "Self-scratching",
    "Sleeping",
    "Walking",
]
ID_COLS = ["Ind_ID", "burst_id", "new_burst"]
TZ = pytz.timezone("Africa/Johannesburg")

# Schema for the per-model prediction files
parquet_schema = pa.schema(
    [
        ("Ind_ID", pa.string()),
        ("burst_id", pa.int32()),
        ("new_burst", pa.int32()),
        ("burst_start_time", pa.timestamp("ns", tz="Africa/Johannesburg")),
        ("predicted_behaviour", pa.string()),
    ]
    + [(c, pa.float32()) for c in CLASS_COLS]
)


def day_bounds(date):
    """Timezone-aware start and end of a day, for parquet row filters."""
    start_dt = TZ.localize(datetime.datetime.combine(date, datetime.time(0, 0, 0)))
    end_dt = TZ.localize(datetime.datetime.combine(date, datetime.time(23, 59, 59)))
    return start_dt, end_dt


def unique_dates(pq_file_path):
    """Sorted unique dates of burst_start_time in a parquet file."""
    df = pd.read_parquet(pq_file_path, columns=["burst_start_time"])
    dates = sorted(pd.to_datetime(df["burst_start_time"]).dt.date.unique())
    print(f"Found {len(dates)} unique dates in {os.path.basename(pq_file_path)}")
    return dates


def format_and_save(final_df, class_names, output_parquet):
    """Fix column types and order, and save a per-model prediction file."""
    final_df["Ind_ID"] = final_df["Ind_ID"].astype(str)
    final_df["burst_id"] = final_df["burst_id"].astype(int)
    final_df["new_burst"] = final_df["new_burst"].astype(int)
    final_df["predicted_behaviour"] = final_df["predicted_behaviour"].astype(str)
    assert list(class_names) == CLASS_COLS, class_names
    for col in CLASS_COLS:
        final_df[col] = final_df[col].astype(float).round(4)
    final_df = final_df[ID_COLS + ["burst_start_time", "predicted_behaviour"] + CLASS_COLS]
    table = pa.Table.from_pandas(final_df, schema=parquet_schema, preserve_index=False)
    pq.write_table(table, output_parquet, compression="zstd", compression_level=9)
    print(f"Saved {len(final_df)} predictions to {output_parquet}")
    return final_df


## HydraMultiROCKET

In [ ]:
def load_model_cuda(model_dir, model_name):
    """Load a tsai learner and check that all parameters and buffers are on CUDA."""
    learn_gpu = ts.load_learner_all(
        path=model_dir,
        dls_fname=f"{model_name}_dls",
        model_fname=f"{model_name}_model",
        learner_fname=f"{model_name}_learner",
    )
    for name, t in list(learn_gpu.model.named_parameters()) + list(
        learn_gpu.model.named_buffers()
    ):
        if t.device.type != "cuda":
            raise RuntimeError(f"'{name}' is on {t.device}, not cuda.")
    print("All model parameters and buffers are on CUDA.")
    return learn_gpu


def extract_acc_for_inference(pq_file_path, date):
    """Extract all acceleration bursts recorded on one date.

    Returns:
        tuple: (X_torch, unique_ids, unique_bursts) -- X_torch is a CUDA tensor
            (n_bursts, 3, n_samples), unique_ids the burst id per row
            ("Ind_ID_burst_id_new_burst"), unique_bursts the start time per burst
    """
    start_dt, end_dt = day_bounds(date)
    acc_cols = ["X", "Y", "Z"]
    id_cols = ["Ind_ID", "burst_id", "new_burst", "sample_number"]
    filtered_df = pq.read_table(
        pq_file_path,
        filters=[
            ("burst_start_time", ">=", start_dt),
            ("burst_start_time", "<", end_dt),
        ],
        columns=id_cols + acc_cols + ["burst_start_time"],
    ).to_pandas()

    # Drop bursts with more samples than expected for this burst length
    burst_num = int(re.search(r"burst_(\d+)", os.path.basename(pq_file_path)).group(1))
    expected_samples = {1: 138, 2: 69, 3: 47, 4: 34}[burst_num]
    burst_sample_counts = (
        filtered_df.groupby(["Ind_ID", "new_burst"])["sample_number"].max().reset_index()
    )
    bad_bursts = burst_sample_counts[burst_sample_counts["sample_number"] > expected_samples]
    if not bad_bursts.empty:
        filtered_df = filtered_df.merge(
            bad_bursts[["Ind_ID", "new_burst"]],
            on=["Ind_ID", "new_burst"],
            how="left",
            indicator=True,
        )
        filtered_df = filtered_df[filtered_df["_merge"] == "left_only"].drop(
            columns=["_merge"]
        )
        print(f"WARNING: removed {len(bad_bursts)} bursts with too many samples on {date}")

    unique_bursts = filtered_df[
        ["Ind_ID", "new_burst", "burst_start_time"]
    ].drop_duplicates()
    unique_bursts["new_burst"] = unique_bursts["new_burst"].astype(str)

    # Long -> wide (one row per burst and axis), then to a tsai array
    melted = pd.melt(
        filtered_df.drop(columns=["burst_start_time"]),
        id_vars=id_cols,
        value_vars=acc_cols,
        var_name="feature",
        value_name="value",
    )
    melted["id"] = (
        melted["Ind_ID"]
        + "_"
        + melted["burst_id"].astype(str)
        + "_"
        + melted["new_burst"].astype(str)
    )
    melted = melted.drop(columns=["Ind_ID", "burst_id", "new_burst"])
    melted = melted.sort_values(by=["id", "sample_number"]).reset_index(drop=True)
    wide = melted.pivot_table(index=["id", "feature"], columns="sample_number", values="value")
    wide.columns = [str(col) for col in wide.columns]
    wide = wide.reset_index()
    unique_ids = wide["id"].unique()
    X, _ = ts.df2xy(wide, sample_col="id", feat_col="feature", target_col=None, data_cols=None)
    X_torch = torch.from_numpy(X.astype(np.float64)).float().cuda()

    n_nan = int(torch.isnan(X_torch).flatten(1).any(dim=1).sum())
    if n_nan:
        print(f"WARNING: {n_nan} bursts with NaNs on {date}")
    return X_torch, unique_ids, unique_bursts


def predict_hr(model, X_torch, unique_ids, unique_bursts, class_names):
    """Predict one date's bursts and attach burst IDs and start times."""
    probas, _, _ = model.get_X_preds(X_torch)
    probas = pd.DataFrame(probas.cpu().numpy(), columns=class_names)
    probas["predicted_behaviour"] = probas[class_names].idxmax(axis=1)
    probas[ID_COLS] = pd.Series(unique_ids).str.split("_", expand=True).to_numpy()
    return probas.merge(unique_bursts, on=["Ind_ID", "new_burst"], how="left")

In [ ]:
hr = load_model_cuda(HR_DIR, HR_NAME)
hr_classes = list(pd.read_csv(f"{HR_DIR}/{HR_NAME}_class_mapping.csv")["class_name"])

hr_list = []
for date in tqdm(unique_dates(FOCAL_ACC), desc="HR: processing dates"):
    X_torch, unique_ids, unique_bursts = extract_acc_for_inference(FOCAL_ACC, date)
    hr_list.append(predict_hr(hr, X_torch, unique_ids, unique_bursts, hr_classes))

hr_pred = format_and_save(pd.concat(hr_list, ignore_index=True), hr_classes, HR_OUT)


## Random Forest

In [ ]:
rf = joblib.load(f"{RF_DIR}/{RF_NAME}_rf_model.joblib")
rf_encoder = joblib.load(f"{RF_DIR}/{RF_NAME}_label_encoder.joblib")
rf_classes = list(rf_encoder.classes_)
# Feature order the model was trained with (sklearn rejects a different order)
rf_features = list(rf.feature_names_in_)

rf_list = []
for date in tqdm(unique_dates(FOCAL_FEATURES), desc="RF: processing dates"):
    start_dt, end_dt = day_bounds(date)
    df = pq.read_table(
        FOCAL_FEATURES,
        filters=[
            ("burst_start_time", ">=", start_dt),
            ("burst_start_time", "<", end_dt),
        ],
    ).to_pandas()
    if df.empty:
        continue
    X = df[rf_features]
    n_nan = int(X.isna().any(axis=1).sum())
    if n_nan:
        print(f"WARNING: {n_nan} bursts with missing features on {date}")
    probas = pd.DataFrame(rf.predict_proba(X), columns=rf_classes)
    probas["predicted_behaviour"] = probas[rf_classes].idxmax(axis=1)
    rf_list.append(
        pd.concat(
            [df[ID_COLS + ["burst_start_time"]].reset_index(drop=True), probas], axis=1
        )
    )

rf_pred = format_and_save(pd.concat(rf_list, ignore_index=True), rf_classes, RF_OUT)

## HydraMultiROCKET, B2 rotbasal and rotdaily
HR B2 rotdaily is the final classifier; rotbasal is a comparison model. Same extraction and prediction as the uncorrected HR above; only the model and the focal-sampled acceleration file differ. Needs the import, constants and HR function cells, not the uncorrected HR run.

In [ ]:
for correction in ["rotbasal", "rotdaily"]:
    name = f"final_hr_b2_{correction}"
    model_dir = f"../models/{name}"
    focal_acc = (
        f"../data/raw/focal_sampling/acc/focal_sampled_acc_burst_2_{correction}.parquet"
    )
    hr_corr = load_model_cuda(model_dir, name)
    corr_classes = list(pd.read_csv(f"{model_dir}/{name}_class_mapping.csv")["class_name"])

    corr_list = []
    for date in tqdm(unique_dates(focal_acc), desc=f"{name}: processing dates"):
        X_torch, unique_ids, unique_bursts = extract_acc_for_inference(focal_acc, date)
        corr_list.append(predict_hr(hr_corr, X_torch, unique_ids, unique_bursts, corr_classes))

    format_and_save(
        pd.concat(corr_list, ignore_index=True),
        corr_classes,
        f"{OUT_DIR}/{name}_focal_sampled_random_predictions.parquet",
    )
    del hr_corr
    torch.cuda.empty_cache()